# Recall-First 3-Class Pipeline V3

Research-only. Non-commercial use only. Not for diagnosis, treatment, or clinical deployment. Doctor review required.

This notebook implements the V3 pipeline:
1. **3-class triage labels** (infectious / non_urgent_dermatologic / referral_urgent)
2. **Recall-first training** — checkpoint on macro recall + referral_urgent recall
3. **Backbone benchmark** — ConvNeXt, ResNet50, EffNet-B0, Swin-Tiny
4. **Temperature calibration** and per-class threshold tuning
5. **Fusion evaluation** — image-only vs image + simulated history (A/B/C)
6. **Optional H2O AutoML** on embeddings + metadata

Writes to `outputs_v3/` (does not overwrite V2).


## 1. Runtime

In Colab: Runtime -> Change runtime type -> GPU.


In [ ]:
!nvidia-smi


## 2. Drive and constants


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_PROJECT = '/content/drive/MyDrive/derm-opd-triage'
REPO_URL = 'https://github.com/Abhigyan-Shekhar/skin-lesion-detect-model.git'
REPO_DIR = '/content/skin-lesion-detect-model'
DATAVERSE_PID = 'doi:10.7910/DVN/W7OUZM'
OUTPUT_ROOT = 'outputs_v3'
CLASS_MAP_PATH = 'configs/class_map_3way.yaml'
BENCHMARK_CONFIG = 'configs/backbone_benchmark_3way.yaml'
RUN_H2O_BENCHMARK = False
RUN_FULL_BACKBONE_BENCHMARK = True
BACKBONES_TO_TRAIN = ['convnext_tiny', 'resnet50', 'efficientnet_b0']

for p in [DRIVE_PROJECT, f'{DRIVE_PROJECT}/data/raw', f'{DRIVE_PROJECT}/{OUTPUT_ROOT}']:
    Path(p).mkdir(parents=True, exist_ok=True)
print('Drive ready:', DRIVE_PROJECT)


## 3. Clone repo and install


In [ ]:
import os, subprocess
from pathlib import Path

def run_checked(cmd, cwd=None):
    print('+', ' '.join(cmd))
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if r.stdout: print(r.stdout)
    if r.returncode != 0:
        if r.stderr: print(r.stderr)
        raise RuntimeError(' '.join(cmd))
    return r

if not Path(REPO_DIR).exists():
    run_checked(['git', 'clone', REPO_URL, REPO_DIR])
else:
    run_checked(['git', 'pull'], cwd=REPO_DIR)

%cd {REPO_DIR}
!pip install -q -r requirements.txt
if RUN_H2O_BENCHMARK:
    !pip install -q h2o


## 4. Link or download data


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, 'src')

DRIVE_RAW = Path(DRIVE_PROJECT) / 'data' / 'raw'
REPO_RAW = Path('data/raw')
REPO_RAW.mkdir(parents=True, exist_ok=True)

metadata_candidates = list(DRIVE_RAW.rglob('Skin_Metadata.csv')) + list(DRIVE_RAW.rglob('skin_metadata.csv'))
image_dirs = [p for p in DRIVE_RAW.rglob('*') if p.is_dir() and p.name.upper() == 'DATASET']

if not metadata_candidates or not image_dirs:
    print('Cache missing — run download_dataverse.py or place files in Drive data/raw/')
else:
    meta_src = metadata_candidates[0]
    img_src = image_dirs[0]
    meta_dst = REPO_RAW / 'METADATA' / 'Skin_Metadata.csv'
    img_dst = REPO_RAW / 'DATASET'
    meta_dst.parent.mkdir(parents=True, exist_ok=True)
    if not meta_dst.exists():
        meta_dst.symlink_to(meta_src)
    if not img_dst.exists():
        img_dst.symlink_to(img_src)
    print('Linked metadata:', meta_dst)
    print('Linked images:', img_dst)


## 5. Coarse labels and patient-level splits


In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

# --- Sys.path and cwd setup (robust to kernel restarts) ---
_SRC_DIR = os.path.join(REPO_DIR, 'src')
if not os.path.isdir(_SRC_DIR):
    raise RuntimeError(f"src/ not found at {_SRC_DIR} — re-run the clone + %cd cell first")
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)
os.chdir(REPO_DIR)

# --- Download or link DermaCon-IN dataset (ported from V2 notebook) ---
def _run_checked(cmd, cwd=None):
    print('+', ' '.join(cmd))
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if r.stdout: print(r.stdout)
    if r.returncode != 0:
        if r.stderr: print(r.stderr)
        raise RuntimeError(' '.join(cmd))
    return r

drive_raw = Path(DRIVE_PROJECT) / 'data' / 'raw'
repo_data = Path(REPO_DIR) / 'data'
repo_raw = repo_data / 'raw'
repo_splits = repo_data / 'splits'
repo_data.mkdir(parents=True, exist_ok=True)
repo_splits.mkdir(parents=True, exist_ok=True)

metadata_path = drive_raw / 'METADATA' / 'Skin_Metadata.csv'
image_dir = drive_raw / 'DATASET'
def _count_images(d: Path) -> int:
    return sum(1 for p in d.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}) if d.exists() else 0

image_count = _count_images(image_dir)
if metadata_path.exists() and image_count > 0:
    print('Using existing Drive dataset cache:', drive_raw)
    print('Images found:', image_count)
else:
    print('Drive dataset cache is incomplete. Downloading DermaCon-IN from Harvard Dataverse...')
    drive_raw.mkdir(parents=True, exist_ok=True)
    _run_checked(['python', 'src/download_dataverse.py',
                  '--persistent_id', DATAVERSE_PID,
                  '--output_dir', str(drive_raw)])
    image_count = _count_images(image_dir)

if not metadata_path.exists():
    raise FileNotFoundError(
        f'Metadata missing after download: {metadata_path}. '
        'If Dataverse returned 403, open the dataset page in a browser and manually upload '
        'Skin_Metadata.csv to this Drive path.'
    )
if image_count == 0:
    raise FileNotFoundError(
        f'No images found after download: {image_dir}. '
        'If Dataverse blocks bulk download, manually upload the .jpg/.png files to this Drive folder.'
    )

# Replace any existing data/raw symlink or directory with a fresh symlink to Drive
if repo_raw.exists() or repo_raw.is_symlink():
    if repo_raw.is_symlink():
        repo_raw.unlink()
    else:
        shutil.rmtree(repo_raw)
repo_raw.symlink_to(drive_raw, target_is_directory=True)
print('Linked', repo_raw, '->', drive_raw)
!find data/raw -maxdepth 3 -type f | head -20

# --- Inspect metadata (ported from V2 notebook section 5) ---
print('\n=== Inspecting metadata ===')
_run_checked(['python', 'src/inspect_metadata.py',
              '--metadata', 'data/raw/METADATA/Skin_Metadata.csv',
              '--image_dir', 'data/raw/DATASET'])

# --- Prepare splits ---
print('\n=== Preparing patient-level splits ===')
_run_checked(['python', 'src/prepare_splits.py',
              '--metadata', 'data/raw/METADATA/Skin_Metadata.csv',
              '--output_dir', 'data/splits'])

_summary_path = Path('data/splits/split_summary.json')
if _summary_path.exists():
    print('\n=== split_summary.json ===')
    print(_summary_path.read_text())

# --- Apply coarse 3-way labels ---
from label_groups import apply_coarse_label, coarse_label_distribution, load_class_map
import pandas as pd

print('\n=== Applying coarse 3-way labels ===')
mapping = load_class_map(CLASS_MAP_PATH)
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'data/splits/{split}.csv')
    df = apply_coarse_label(df, target_col='coarse_class', mapping=mapping)
    df.to_csv(f'data/splits/{split}.csv', index=False)
    print(split, coarse_label_distribution(df, 'coarse_class'))

Path(f'{OUTPUT_ROOT}/metrics').mkdir(parents=True, exist_ok=True)
with open(f'{OUTPUT_ROOT}/metrics/coarse_label_distribution.json', 'w') as f:
    json.dump({
        split: coarse_label_distribution(pd.read_csv(f'data/splits/{split}.csv'), 'coarse_class')
        for split in ['train', 'val', 'test']
    }, f, indent=2)
print('\nWrote', f'{OUTPUT_ROOT}/metrics/coarse_label_distribution.json')

## 6. Imports


In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from utils import read_yaml, write_json, ensure_dir
from recall_training import train_v3, evaluate_full_v3
from dataset import DermatologyDataset, build_transforms
from evaluate_fusion import run_fusion_experiments

benchmark_cfg = read_yaml(BENCHMARK_CONFIG)
benchmark_cfg['image_dir'] = 'data/raw/DATASET'
benchmark_cfg['class_map_path'] = CLASS_MAP_PATH
print('Config loaded. Backbones:', BACKBONES_TO_TRAIN)


## 7. Backbone benchmark (recall-first)


In [ ]:
comparison_rows = []
bundles = {}

for model_name in BACKBONES_TO_TRAIN:
    cfg = dict(benchmark_cfg)
    cfg['model_name'] = model_name
    out_dir = f'{OUTPUT_ROOT}/model_{model_name}'
    print(f'\n===== {model_name} =====')
    bundle = train_v3(cfg, out_dir, cfg['train_csv'], cfg['val_csv'], label_column_override='coarse_class')
    bundles[model_name] = bundle

    test_ds = DermatologyDataset(
        csv_path=cfg['test_csv'], image_dir=cfg['image_dir'],
        image_column=bundle['image_column'], label_column=bundle['label_column'],
        class_to_idx=bundle['class_to_idx'],
        transform=build_transforms(int(cfg['image_size']), train=False),
    )
    test_loader = DataLoader(test_ds, batch_size=int(cfg['batch_size']), shuffle=False, num_workers=2)
    metrics = evaluate_full_v3(
        bundle['model'], test_loader, bundle['class_names'], bundle['image_column'],
        bundle['temperature'], bundle['device'], out_dir, tag='test',
        thresholds=bundle['thresholds'],
    )
    comparison_rows.append({'model_name': model_name, **{k: v for k, v in metrics.items() if isinstance(v, float)}})

comparison_df = pd.DataFrame(comparison_rows).sort_values('recall_first_score', ascending=False)
comparison_df.to_csv(f'{OUTPUT_ROOT}/benchmark_comparison.csv', index=False)
display(comparison_df)

best_model_name = comparison_df.iloc[0]['model_name']
best_bundle = bundles[best_model_name]
write_json(f'{OUTPUT_ROOT}/best_model.json', {'model_name': best_model_name, 'output_dir': f'{OUTPUT_ROOT}/model_{best_model_name}'})
print('Best backbone:', best_model_name)


## 8. Fusion evaluation (image vs history)


In [ ]:
test_df = pd.read_csv('data/splits/test.csv')
test_ds = DermatologyDataset(
    csv_path=benchmark_cfg['test_csv'], image_dir=benchmark_cfg['image_dir'],
    image_column=best_bundle['image_column'], label_column=best_bundle['label_column'],
    class_to_idx=best_bundle['class_to_idx'],
    transform=build_transforms(int(benchmark_cfg['image_size']), train=False),
)
test_loader = DataLoader(test_ds, batch_size=int(benchmark_cfg['batch_size']), shuffle=False, num_workers=2)

fusion_lift = run_fusion_experiments(
    best_bundle['model'], test_loader, test_df, best_bundle['class_names'],
    best_bundle['device'], best_bundle['temperature'],
    f'{OUTPUT_ROOT}/model_{best_model_name}', alpha=None,
)
print('Fusion lift:', fusion_lift)


## 9. Optional H2O AutoML benchmark


In [ ]:
if RUN_H2O_BENCHMARK:
    import h2o
    from h2o.automl import H2OAutoML
    from extract_embeddings import extract_embeddings, build_tabular_frame
    from utils import detect_columns, read_table
    from metrics import compute_epoch_metrics

    ckpt = f'{OUTPUT_ROOT}/model_{best_model_name}/checkpoints/best.pt'
    emb_path = f'{OUTPUT_ROOT}/embeddings_test.csv'
    extract_embeddings(
        benchmark_cfg['test_csv'], benchmark_cfg['image_dir'], ckpt, best_model_name,
        best_bundle['label_column'], best_bundle['image_column'], best_bundle['class_to_idx'],
        emb_path, batch_size=32,
    )
    meta = read_table('data/raw/METADATA/Skin_Metadata.csv')
    tab = build_tabular_frame(pd.read_csv(emb_path), 'data/raw/METADATA/Skin_Metadata.csv', detect_columns(meta.columns.tolist()))
    tab['coarse_class'] = tab[best_bundle['label_column']] if best_bundle['label_column'] in tab.columns else tab['label_idx']

    h2o.init()
    hf = h2o.H2OFrame(tab)
    x = [c for c in hf.columns if c.startswith('emb_')]
    y = 'coarse_class'
    train_h, valid_h = hf.split_frame(ratios=[0.8], seed=42)
    aml = H2OAutoML(max_models=10, balance_classes=True, seed=42, sort_metric='AUCPR')
    aml.train(x=x, y=y, training_frame=train_h, validation_frame=valid_h)
    perf = aml.leader.model_performance(valid_h)
    print(perf)
    h2o.cluster().shutdown()
else:
    print('H2O benchmark skipped (RUN_H2O_BENCHMARK=False)')


## 10. Summary


In [ ]:
from utils import DISCLAIMER_TEXT
print('=== V3 Pipeline Complete ===')
print('Best model:', best_model_name)
print('Outputs:', OUTPUT_ROOT)
print('Benchmark:', f'{OUTPUT_ROOT}/benchmark_comparison.csv')
print('Fusion lift:', f'{OUTPUT_ROOT}/model_{best_model_name}/metrics/fusion_lift.json')
print(DISCLAIMER_TEXT)
